<a href="https://colab.research.google.com/github/whitestones011/deep_learning/blob/colab/tokenizer_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets -qU

# Training BPE tokenizer

In [ ]:
from datasets import load_dataset

ds = load_dataset("iohadrubin/wikitext-103-raw-v1")

In [ ]:
ds

**Training Byte-Pair Encoding (BPE) tokenizer**

* Start with all the characters present in the training corpus as tokens.

* Identify the most common pair of tokens and merge it into one token.

* Repeat until the vocabulary (e.g., the number of tokens) has reached the size we want.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE


tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

In [ ]:
from tokenizers.trainers import BpeTrainer

In [ ]:
trainer = BpeTrainer(
    vocab_size=100,
    min_frequency=50000,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

In [ ]:
from tokenizers.pre_tokenizers import Whitespace
tokenizer.pre_tokenizer = Whitespace()

In [ ]:
tokenizer.train(ds['train']['text'], trainer)

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
tokenizer.backend_tokenizer

In [ ]:
tokenizer.backend_tokenizer.normalizer.normalize_str('How  ar,you?')

In [ ]:
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str('How  ar,you?')

# BPE

In [ ]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

GPT-2 tokenizer uses SentencePiece tokenization algo.

In [ ]:
tokenizer.backend_tokenizer.normalizer

In [ ]:
tokenizer.backend_tokenizer.pre_tokenizer

In [ ]:
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(corpus[0])

In [ ]:
# set of unique words in the corpus
# calculate word frequencies

In [ ]:
# WORD FREQUENCIES

from collections import Counter

word_freqs = Counter()

for sentence in corpus:
  # split sentence into (word,offset)
  words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(sentence)
  # get words
  words = [word for word, offset in words_with_offsets]
  word_freqs.update(words)

print(word_freqs)

In [ ]:
word_freqs.most_common(10)

In [ ]:
# ALPHABET
# build alphabet from uniques tokens
alphabet = []

(alphabet := list({char for word in word_freqs.elements() for char in word}))
alphabet.sort()

print(alphabet)

In [ ]:
len(set(alphabet))

In [ ]:
# VOCABULARY
# add special tokens
vocab = ["<|endoftext|>"] + alphabet.copy()

In [ ]:
#  Split each word into individual characters, to be able to start training
_ = (splits := {word: [i for i in word] for word in word_freqs.elements()})

In [ ]:
from collections import defaultdict
from itertools import pairwise

# computes the frequency of each pair
# frequency will be equal to word_freqs
def compute_pair_freqs(split):
  pair_freq = defaultdict(int)
  for word, freq in word_freqs.items():
    split = splits[word]
    if len(split) == 1:
      continue
    for pair in pairwise(split):
      pair_freq[pair] += freq

  return pair_freq

In [ ]:
pair_freqs = compute_pair_freqs(splits)

In [ ]:
for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break

In [ ]:
# most frequent pair
most_freq_pair = max(pair_freqs, key=pair_freqs.get)
most_freq_pair

In [ ]:
len(vocab)

In [ ]:
#  add most frequent pair to vocab
merges = {most_freq_pair: ''.join(most_freq_pair)}
# vocab.append(''.join(most_freq_pair))

In [ ]:
# len(vocab)

In [ ]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

In [ ]:
splits["Ġtrained"]

In [ ]:
most_freq_pair

In [ ]:
def merge_pair(most_freq_pair, splits):
  for word in word_freqs:
    split = splits[word]
    if len(split) == 1:
      continue
    try:
      idx = split.index(most_freq_pair[0])
      if idx != len(split)-1:
        if split[idx+1] == most_freq_pair[1]:
          split = split[:idx] + [''.join(most_freq_pair)] + split[idx+2:]
          splits[word] = split
    except ValueError:
      continue

  return splits

In [ ]:
_ = merge_pair(most_freq_pair, splits)

We are going to merge Ġt in splits dictionary

In [ ]:
splits["Ġtrained"]

In [ ]:
len(splits)

In [ ]:
# VOCAB LOOP
vocab_size = 50
merges = defaultdict(str)

while len(vocab) < vocab_size:
  pair_freqs = compute_pair_freqs(splits) # calculate frequency of pair of characters over all corpus
  most_freq_pair = max(pair_freqs, key=pair_freqs.get) # find most frequent pair
  if most_freq_pair not in merges.keys():
    merges[most_freq_pair] = ''.join(most_freq_pair) # add to merges
    vocab.append(''.join(most_freq_pair)) # add to vocab
  _ = merge_pair(most_freq_pair, splits) # merge pairs

In [ ]:
len(vocab)

In [ ]:
len(set(vocab))

In [ ]:
#  TOKENIZER

def tokenize(text):
    words_with_offsets = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    words = [word for word, offset in words_with_offsets]
    splits = [[i for i in word] for word in words]
    for pair, merge in merges.items():
        for split_idx, split in enumerate(splits):

          if len(split) == 1:
                continue
          try:
            idx = split.index(pair[0])
            if idx != len(split)-1:
              if split[idx+1] == pair[1]:
                split = split[:idx] + [''.join(pair)] + split[idx+2:]
                splits[split_idx] = split
          except ValueError:
            continue

    return sum(splits, [])

In [ ]:
tokenize('This is best grain')

# WordPiece

In [ ]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [ ]:
# pre-tokenize corpus into words using BERT model
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
tokenizer.backend_tokenizer.normalizer

In [ ]:
from collections import Counter

word_freqs = Counter()

# for each word in sentence
for sentence in corpus:
  # split sentence into words and return word with offset
  words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(sentence)
  # select words
  words = [word for word, offset in words_with_offsets]
  # unique words with frequencies
  word_freqs.update(words)


In [ ]:
# ALPHABET
# all first characters and add ## for preceding characters
alphabet = []

(alphabet := list({char if idx==0 else f'##{char}' for word in word_freqs.elements() for idx, char in enumerate(word)}))

alphabet.sort()

print(alphabet)

In [ ]:
len(alphabet)

In [ ]:
tokenizer.all_special_tokens

In [ ]:
# VOCABULARY
# add special tokens and alphabet
vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]",  "[MASK]"] + alphabet.copy()

In [ ]:
# split words in vocabulary into characters
splits = {
    word:[i if idx == 0 else f'##{i}' for idx, i in enumerate(word)]
    for word in word_freqs.elements()
    }

In [ ]:
splits['This']

In [ ]:
# compute pair frequencies

from collections import defaultdict
from itertools import pairwise

def compute_pair_score(splits):
  pair_freq = defaultdict(int)
  char_freq = defaultdict(int)

  for word, freq in word_freqs.items():
    split = splits[word]
    if len(split) == 1:
      char_freq[split[0]] += freq
      continue
    for pair in pairwise(split):
      pair_freq[pair] += freq
      # avoid double counting second char!
      char_freq[pair[0]] += freq

    # add freq to the last char in the word
    char_freq[pair[1]] += freq

  pair_score = {
      pair: freq / (char_freq[pair[0]] * char_freq[pair[1]]) for pair, freq in pair_freq.items()
      }

  return pair_score

pair_scores = compute_pair_score(splits)

In [ ]:
for idx, pair in enumerate(pair_scores):
  if idx <5:
    print(pair, pair_scores[pair])

In [ ]:
# PAIR WITH HIGHEST SCORE
max(pair_scores)

In [ ]:
def merge_pair(pair, splits):
  a, b = pair[0], pair[1]
  for key, split in splits.items():
    if len(split) == 1:
      continue
    try:
      idx = split.index(a)
      #  if not the last
      if idx != len(split)-1:
        if split[idx+1] == b:
          merge_str =  a + b[2:] if b.startswith("##") else a + b
          split = split[:idx] + [merge_str] + split[idx+2:]
          splits[key] = split
    except ValueError:
      continue

  return splits

In [ ]:
# VOCAB LOOP
vocab_size = 70
merges = defaultdict(str)

while len(vocab) < vocab_size:
  pair_scores = compute_pair_score(splits) # calculate frequency of pair of characters over all corpus
  best_pair = max(pair_scores, key=pair_scores.get) # find most frequent pair
  new_token = (
        best_pair[0] + best_pair[1][2:]
        if best_pair[1].startswith("##")
        else best_pair[0] + best_pair[1]
    )
  merges[best_pair] = new_token # add to merges
  vocab.append(new_token) # add to vocab
  _ = merge_pair(best_pair, splits) # merge best pair in spits

In [ ]:
len(vocab)

In [ ]:
merges

In [ ]:
# encode word into tokens starting with the longest
def encode_word(word):
  tokens = []
  while len(word) > 0:
    i = 1
    while i < len(word) and word[:i] not in vocab:
      i += 1
    word = word[i:]


In [ ]:
def encode_word(word):
  tokens = []
  ii = 0
  while len(word)>0:
    i = len(word)
    #  scan from the end of the string
    while i >0 and word[:i] not in vocab:
      i -= 1
    # encode whole word as UNK if substrings not found
    if i == 0:
        return ["[UNK]"]
    tokens.append(word[:i])
    word = word[i:]
    if len(word) > 0:
        word = f"##{word}"
  return tokens

In [ ]:
encode_word('Hugging')

In [ ]:
# encoding
def tokenize_text(text):
  words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
  encoded_words = [encode_word(word) for word, offset in words_with_offsets]
  return sum(encoded_words, [])

In [ ]:
sum([['a'], ['b']], [])

In [ ]:
tokenize_text("This is the Hugging Face course!")

# Unigram tokenization

# Tokenizer from scratch

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("wikitext", name="wikitext-2-raw-v1", split="train")

In [ ]:
dataset

In [ ]:
def get_training_corpus(dataset, n):
  for i in range(0, len(dataset), n):
    yield dataset[i : i + n]["text"]

In [ ]:
generator = get_training_corpus(dataset, 10_000)

Building WordPiece tokenizer

In [ ]:
from tokenizers import (
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    models,
    decoders,
    Tokenizer
    )

In [ ]:
tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))

In [ ]:
tokenizer.normalizer = normalizers.Sequence(
    [normalizers.NFD(), normalizers.Lowercase(), normalizers.StripAccents()]
    )

In [ ]:
tokenizer.pre_tokenizer = pre_tokenizers.Sequence(
    [pre_tokenizers.WhitespaceSplit(), pre_tokenizers.Punctuation() ]
)

In [ ]:
tokenizer.pre_tokenizer.pre_tokenize_str("How are you?")

In [ ]:
trainer = trainers.WordPieceTrainer(
    vocab_size=25000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

In [ ]:
# training tokenizer
tokenizer.train_from_iterator(generator, trainer=trainer)

In [ ]:
encoded_str = tokenizer.encode("How are you?")

In [ ]:
encoded_str

In [ ]:
encoded_str.tokens

In [ ]:
# post-processing - add [CLS] , [SEP]
cls_token_id = tokenizer.token_to_id("[CLS]")
sep_token_id = tokenizer.token_to_id("[SEP]")
print(cls_token_id, sep_token_id)

In [ ]:
tokenizer.post_processor = processors.TemplateProcessing(
    single=f"[CLS]:0 $A:0 [SEP]:0",
    pair=f"[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
    special_tokens=[("[CLS]", cls_token_id), ("[SEP]", sep_token_id)],
)

In [ ]:
encoding = tokenizer.encode("Let's test this tokenizer.")

In [ ]:
encoding.tokens

In [ ]:
tokenizer.decoder = decoders.WordPiece(prefix="##")

In [ ]:
encoding.ids

In [ ]:
tokenizer.decode(encoding.ids)

In [ ]:
tokenizer.save('tokenizer.json')